# KYNTRA_02_BASELINE — Event-Grouped Logistic Regression

Purpose: build the first real KYNTRA overtake baseline using TRAIN events only.

Primary target: `P(pass within 1 lap)`

This notebook:
- uses leave-one-event-out (LOEO) CV across the 7 TRAIN races
- compares climatology, gap-only, dynamics, and contextual Logistic Regression
- evaluates global and tactical battle slices
- tests battle-sequence weighting as an ablation
- saves event-held-out OOF predictions

It does NOT touch demo holdouts, use VALIDATION for selection, train retention ML, or relabel censored rows as negatives.


In [ ]:
!pip -q install pyarrow scikit-learn

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, precision_score,
    recall_score, f1_score, roc_auc_score, precision_recall_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.calibration import calibration_curve

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Paths


In [ ]:
ROOT = Path("/content/drive/MyDrive/KYNTRA")
DATA = ROOT / "datasets"
EXPERIMENTS = ROOT / "experiments" / "BASELINE_02"
EXPERIMENTS.mkdir(parents=True, exist_ok=True)

PRIMARY = DATA / "kyntra_overtake_dataset.parquet"
assert PRIMARY.exists(), f"Dataset not found: {PRIMARY}"


## Load locked dataset


In [ ]:
df = pd.read_parquet(PRIMARY)

assert df.shape[0] == 8357
assert df["observation_id"].is_unique
assert set(df["split"].unique()) == {"TRAIN", "VALIDATION"}

print("Dataset:", df.shape)
print(df["split"].value_counts())


## Modeling population

For this baseline, censored 1-lap rows are excluded. They are not relabeled as zero.

This is a pragmatic complete-case classification baseline under potentially informative censoring; we are NOT assuming censoring is random.


In [ ]:
TARGET = "overtake_next_1_lap"
CENSOR = "censored_1_lap"

train_all = df[df["split"] == "TRAIN"].copy()
validation_frozen = df[df["split"] == "VALIDATION"].copy()

train = train_all[train_all[CENSOR] != 1].copy()

assert len(train) == 5631
assert int(train[TARGET].sum()) == 181

print("TRAIN all:", train_all.shape)
print("TRAIN usable 1L:", train.shape)
print("TRAIN positives:", int(train[TARGET].sum()))
print("TRAIN prevalence:", float(train[TARGET].mean()))
print("Validation frozen:", validation_frozen.shape)


## Transparent feature engineering


In [ ]:
train["log_gap_seconds"] = np.log1p(train["gap_seconds"].clip(lower=0))

FEATURE_SETS = {
    "gap_only": {
        "numeric": ["log_gap_seconds"],
        "categorical": [],
    },
    "dynamics": {
        "numeric": [
            "log_gap_seconds",
            "closing_rate",
            "recent_pace_delta_1lap",
            "recent_pace_delta_3laps",
            "speed_trap_delta",
        ],
        "categorical": [],
    },
    "context": {
        "numeric": [
            "log_gap_seconds",
            "closing_rate",
            "recent_pace_delta_1lap",
            "recent_pace_delta_3laps",
            "speed_trap_delta",
            "tyre_age_delta",
            "attacker_position",
            "defender_position",
            "sector1_delta",
            "sector2_delta",
            "consecutive_laps_following",
            "consecutive_laps_close",
            "laps_remaining",
            "distance_gap_mean_recent",
            "distance_gap_std_recent",
            "rear_distance_gap_m",
        ],
        "categorical": [
            "attacker_compound",
            "defender_compound",
            "race_phase",
            "rear_threat_proxy",
            "track_status_parsed",
            "weather_rainfall",
        ],
    },
}

for cfg in FEATURE_SETS.values():
    cfg["numeric"] = [c for c in cfg["numeric"] if c in train.columns]
    cfg["categorical"] = [c for c in cfg["categorical"] if c in train.columns]

for name, cfg in FEATURE_SETS.items():
    print("\n", name)
    print("numeric:", cfg["numeric"])
    print("categorical:", cfg["categorical"])


## Leakage guardrail


In [ ]:
PROHIBITED = {
    "Driver","DriverNumber","Team","attacker","defender","attacker_team","defender_team",
    "event_id","race_id","event_name","circuit","season","split",
    "battle_sequence_id","observation_id",
    "overtake_next_1_lap","overtake_next_2_laps","overtake_next_3_laps",
    "censored_1_lap","censored_2_laps","censored_3_laps",
    "overtake_event_id","overtake_event_lap","overtake_event_time",
    "overtake_event_attacker","overtake_event_defender",
    "overtake_verification_method","overtake_confidence",
    "label_source","label_version","label_generated_at",
}

for name, cfg in FEATURE_SETS.items():
    features = set(cfg["numeric"] + cfg["categorical"])
    bad = sorted(features & PROHIBITED)
    assert not bad, f"{name} contains prohibited fields: {bad}"

print("Leakage audit: PASS")


## Model pipeline


In [ ]:
def build_pipeline(numeric_features, categorical_features):
    transformers = []

    if numeric_features:
        num = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", RobustScaler()),
        ])
        transformers.append(("num", num, numeric_features))

    if categorical_features:
        cat = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("cat", cat, categorical_features))

    preprocess = ColumnTransformer(transformers, remainder="drop")

    model = LogisticRegression(
        max_iter=2500,
        solver="liblinear",
        class_weight=None,
        random_state=42,
    )

    return Pipeline([
        ("preprocess", preprocess),
        ("model", model),
    ])


## Metrics helpers


In [ ]:
def ece_quantile(y_true, p, n_bins=10):
    tmp = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(p)})
    try:
        tmp["bin"] = pd.qcut(tmp["p"], q=n_bins, duplicates="drop")
    except ValueError:
        return np.nan

    ece = 0.0
    for _, g in tmp.groupby("bin", observed=True):
        if len(g):
            ece += (len(g)/len(tmp)) * abs(g["y"].mean() - g["p"].mean())
    return float(ece)

def metric_dict(y_true, p, climatology_p=None):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)

    out = {
        "n": len(y_true),
        "positives": int(y_true.sum()),
        "prevalence": float(y_true.mean()),
        "pr_auc_ap": float(average_precision_score(y_true, p)),
        "brier": float(brier_score_loss(y_true, p)),
        "ece_q10": ece_quantile(y_true, p, 10),
        "roc_auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) == 2 else np.nan,
    }

    if climatology_p is not None:
        clim_bs = brier_score_loss(y_true, climatology_p)
        out["climatology_brier"] = float(clim_bs)
        out["brier_skill_score"] = float(1 - out["brier"]/clim_bs) if clim_bs > 0 else np.nan

    for thr in [0.05,0.10,0.15,0.20,0.30]:
        pred = (p >= thr).astype(int)
        out[f"precision@{thr:.2f}"] = float(precision_score(y_true, pred, zero_division=0))
        out[f"recall@{thr:.2f}"] = float(recall_score(y_true, pred, zero_division=0))
        out[f"f1@{thr:.2f}"] = float(f1_score(y_true, pred, zero_division=0))
    return out


## Battle-sequence sample weighting ablation


In [ ]:
def make_sample_weights(fold_train, policy):
    if policy == "uniform":
        return np.ones(len(fold_train), dtype=float)

    if policy == "inv_sqrt_sequence":
        lengths = fold_train.groupby("battle_sequence_id").size()
        w = fold_train["battle_sequence_id"].map(
            lambda s: 1.0 / np.sqrt(lengths.loc[s])
        ).astype(float).to_numpy()
        return w / w.mean()

    raise ValueError(policy)


## Leave-One-Event-Out OOF runner


In [ ]:
def run_loeo(feature_set_name, weighting_policy):
    cfg = FEATURE_SETS[feature_set_name]
    numeric = cfg["numeric"]
    categorical = cfg["categorical"]
    features = numeric + categorical

    oof_parts = []
    fold_rows = []

    for holdout_event in sorted(train["event_id"].unique()):
        fold_train = train[train["event_id"] != holdout_event].copy()
        fold_test = train[train["event_id"] == holdout_event].copy()

        X_train = fold_train[features]
        y_train = fold_train[TARGET].astype(int)
        X_test = fold_test[features]
        y_test = fold_test[TARGET].astype(int)

        pipe = build_pipeline(numeric, categorical)
        weights = make_sample_weights(fold_train, weighting_policy)

        pipe.fit(X_train, y_train, model__sample_weight=weights)
        p = pipe.predict_proba(X_test)[:,1]

        # Fold-specific climatology uses only the six training events.
        p_clim = np.full(len(fold_test), y_train.mean(), dtype=float)

        part = fold_test[[
            "observation_id","event_id","battle_sequence_id","gap_seconds",TARGET
        ]].copy()
        part["pred"] = p
        part["climatology_pred"] = p_clim
        oof_parts.append(part)

        m = metric_dict(y_test, p, p_clim)
        m["holdout_event"] = holdout_event
        m["feature_set"] = feature_set_name
        m["weighting"] = weighting_policy
        fold_rows.append(m)

    return pd.concat(oof_parts, ignore_index=True), pd.DataFrame(fold_rows)


## E0 — Climatology baseline


In [ ]:
clim_parts = []

for holdout_event in sorted(train["event_id"].unique()):
    fold_train = train[train["event_id"] != holdout_event]
    fold_test = train[train["event_id"] == holdout_event].copy()
    p0 = fold_train[TARGET].mean()

    part = fold_test[[
        "observation_id","event_id","battle_sequence_id","gap_seconds",TARGET
    ]].copy()
    part["pred"] = p0
    part["climatology_pred"] = p0
    clim_parts.append(part)

clim_oof = pd.concat(clim_parts, ignore_index=True)
display(pd.DataFrame([metric_dict(
    clim_oof[TARGET],
    clim_oof["pred"],
    clim_oof["climatology_pred"],
)]))


## E1–E3 — Logistic baseline ladder


In [ ]:
results = {}
summary_rows = []

for feature_set in ["gap_only","dynamics","context"]:
    oof, folds = run_loeo(feature_set, "uniform")
    results[(feature_set,"uniform")] = {"oof":oof, "folds":folds}

    pooled = metric_dict(oof[TARGET], oof["pred"], oof["climatology_pred"])
    pooled.update({
        "feature_set":feature_set,
        "weighting":"uniform",
        "macro_fold_pr_auc":folds["pr_auc_ap"].mean(),
        "std_fold_pr_auc":folds["pr_auc_ap"].std(),
    })
    summary_rows.append(pooled)

baseline_summary = pd.DataFrame(summary_rows)
display(baseline_summary[[
    "feature_set","pr_auc_ap","macro_fold_pr_auc","std_fold_pr_auc",
    "brier","brier_skill_score","ece_q10","roc_auc"
]].sort_values("pr_auc_ap", ascending=False))


## Per-event metrics


In [ ]:
for key, obj in results.items():
    print("\n###", key)
    display(obj["folds"][[
        "holdout_event","n","positives","prevalence",
        "pr_auc_ap","brier","brier_skill_score","ece_q10","roc_auc"
    ]].sort_values("holdout_event"))


## E4 — Sequence-weighting ablation


In [ ]:
oof_w, folds_w = run_loeo("context", "inv_sqrt_sequence")
results[("context","inv_sqrt_sequence")] = {"oof":oof_w, "folds":folds_w}

rows = []
for key in [("context","uniform"),("context","inv_sqrt_sequence")]:
    oof = results[key]["oof"]
    folds = results[key]["folds"]
    pooled = metric_dict(oof[TARGET], oof["pred"], oof["climatology_pred"])
    pooled.update({
        "feature_set":"context",
        "weighting":key[1],
        "macro_fold_pr_auc":folds["pr_auc_ap"].mean(),
        "std_fold_pr_auc":folds["pr_auc_ap"].std(),
    })
    rows.append(pooled)

weighting_summary = pd.DataFrame(rows)
display(weighting_summary[[
    "feature_set","weighting","pr_auc_ap","macro_fold_pr_auc","std_fold_pr_auc",
    "brier","brier_skill_score","ece_q10","roc_auc"
]])


## Global vs tactical-slice evaluation

These are TRAIN-derived evaluation slices, not hard training filters:
- <=1.0s very close battle
- <=1.5s captures ~92.8% of TRAIN imminent passes
- <=3.0s captures ~98.3% of TRAIN imminent passes


In [ ]:
def evaluate_slices(oof, model_name):
    rows = []
    masks = [
        ("GLOBAL", np.ones(len(oof), dtype=bool)),
        ("GAP_LE_1.0S", oof["gap_seconds"] <= 1.0),
        ("GAP_LE_1.5S", oof["gap_seconds"] <= 1.5),
        ("GAP_LE_3.0S", oof["gap_seconds"] <= 3.0),
    ]

    for label, mask in masks:
        part = oof.loc[mask].copy()
        if not len(part):
            continue
        m = metric_dict(part[TARGET], part["pred"], part["climatology_pred"])
        m["model"] = model_name
        m["slice"] = label
        rows.append(m)
    return pd.DataFrame(rows)

slice_summary = pd.concat([
    evaluate_slices(results[key]["oof"], f"{key[0]}__{key[1]}")
    for key in [
        ("gap_only","uniform"),
        ("dynamics","uniform"),
        ("context","uniform"),
        ("context","inv_sqrt_sequence"),
    ]
], ignore_index=True)

display(slice_summary[[
    "model","slice","n","positives","prevalence","pr_auc_ap",
    "brier","brier_skill_score","ece_q10",
    "precision@0.10","recall@0.10","precision@0.20","recall@0.20"
]])


## Proximity-confounder test

The key scientific question:

Does dynamics/context materially outperform gap-only inside the close tactical slices?

If not, KYNTRA has not yet demonstrated enough intelligence beyond proximity.


In [ ]:
display(
    slice_summary.pivot_table(
        index="slice",
        columns="model",
        values="pr_auc_ap"
    )
)


## Precision–Recall curves — TRAIN OOF only


In [ ]:
plt.figure(figsize=(9,6))

for key in [
    ("gap_only","uniform"),
    ("dynamics","uniform"),
    ("context","uniform"),
    ("context","inv_sqrt_sequence"),
]:
    oof = results[key]["oof"]
    precision, recall, _ = precision_recall_curve(oof[TARGET], oof["pred"])
    plt.plot(recall, precision, label=f"{key[0]} / {key[1]}")

plt.axhline(train[TARGET].mean(), linestyle="--", label="TRAIN prevalence")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("KYNTRA — LOEO OOF Precision–Recall")
plt.legend()
plt.show()


## Reliability curves — TRAIN OOF only


In [ ]:
plt.figure(figsize=(7,7))

for key in [
    ("gap_only","uniform"),
    ("dynamics","uniform"),
    ("context","uniform"),
    ("context","inv_sqrt_sequence"),
]:
    oof = results[key]["oof"]
    prob_true, prob_pred = calibration_curve(
        oof[TARGET], oof["pred"], n_bins=10, strategy="quantile"
    )
    plt.plot(prob_pred, prob_true, marker="o", label=f"{key[0]} / {key[1]}")

plt.plot([0,1],[0,1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed frequency")
plt.title("KYNTRA — LOEO OOF Reliability")
plt.legend()
plt.show()


## Save artifacts


In [ ]:
for key, obj in results.items():
    feature_set, weighting = key
    obj["oof"].to_csv(EXPERIMENTS / f"oof_{feature_set}_{weighting}.csv", index=False)
    obj["folds"].to_csv(EXPERIMENTS / f"fold_metrics_{feature_set}_{weighting}.csv", index=False)

baseline_summary.to_csv(EXPERIMENTS / "baseline_summary.csv", index=False)
weighting_summary.to_csv(EXPERIMENTS / "weighting_summary.csv", index=False)
slice_summary.to_csv(EXPERIMENTS / "tactical_slice_summary.csv", index=False)

print("Saved to:", EXPERIMENTS)


# STOP — DO NOT TOUCH VALIDATION YET

Send ChatGPT:
1. `baseline_summary`
2. context-model per-event fold metrics
3. `weighting_summary`
4. `slice_summary`
5. PR curve screenshot
6. reliability curve screenshot

Only after review do we create `KYNTRA_03_MODEL_V1.ipynb`.
